# Les thèmes de la presse de 1914

De quoi parlait *L'Humanité* en 1914, et comment l'actualité a-t-elle basculé l'été venu ? Sans aucune étiquette, BERTopic regroupe les passages proches et décrit chaque groupe par ses mots les plus caractéristiques.

1. Trouver les thèmes
2. Les ajuster
3. Les suivre dans le temps

Les *embeddings* des passages sont déjà calculés (avec Qwen3-Embedding-0.6B, tronqués à 512 dimensions) : ce notebook n'a pas besoin de GPU.

## Données

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/embeddings-512.npy
%pip install -q "bertopic<1"

In [ ]:
import numpy as np
import pandas as pd

passages = pd.read_parquet("passages.parquet")
embeddings = np.load("embeddings-512.npy").astype(np.float32)

# L'Humanité seulement (Le Figaro n'a que 24 numéros), et un échantillon de 20 000
# passages pour que le calcul tienne en quelques minutes.
humanite = passages[passages.newspaper == "L'Humanité"].sample(20_000, random_state=0)
docs = humanite.text.tolist()
dates = humanite.date.tolist()
vectors = embeddings[humanite.index]
print(len(docs), vectors.shape)

## 1. Trouver les thèmes

Créez un modèle `BERTopic` qui réduit les *embeddings* à 5 dimensions avec UMAP (de façon reproductible), décrit les thèmes avec un `CountVectorizer` sans mots vides (pour **décrire** les thèmes, ils gênent : c'est le seul usage du cours où on les retire), et ne garde que les thèmes d'au moins 50 passages. Le corpus est en français.

Entraînez-le avec `fit_transform` (sur les vecteurs déjà calculés), puis affichez `get_topic_info()` : combien de thèmes ? Quelle part des passages est hors thème (thème $-1$) ? Lisez les mots-clés des 15 plus grands thèmes, et deux passages représentatifs (`get_representative_docs`) de trois d'entre eux.

**Documentation utile**

- [`BERTopic`](https://maartengr.github.io/BERTopic/api/bertopic.html)
- [`BERTopic.fit_transform`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.fit_transform)
- [`BERTopic.get_topic_info`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_topic_info)
- [`BERTopic.get_representative_docs`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.get_representative_docs)
- [Paramètres d'UMAP](https://umap-learn.readthedocs.io/en/latest/parameters.html)
- [`CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from spacy.lang.fr.stop_words import STOP_WORDS
from umap import UMAP

STOP_WORDS = sorted(STOP_WORDS | {"qu", "mm", "mme"})

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0)` : une réduction à 5 dimensions, reproductible
- `CountVectorizer(stop_words=STOP_WORDS, min_df=10)` : `min_df=10` ignore les mots présents dans moins de 10 passages
- `BERTopic(language="french", umap_model=..., vectorizer_model=..., min_topic_size=50)` : sans `language="french"`, BERTopic retire des mots-clés tous les caractères accentués (« grève » devient « grve »)
- `fit_transform(docs, vectors)` : le second argument évite de recalculer les *embeddings* ; renvoie les thèmes de chaque passage, puis des probabilités
- `get_topic_info()` : une ligne par thème, le thème −1 étant celui des passages hors thème ; `np.mean(np.array(topics) == -1)` : leur part
- `get_topic(thème)` : les mots-clés d'un thème et leurs poids ; `get_representative_docs(thème)` : ses passages représentatifs

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
topic_model = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=...,
)
topics, _ = topic_model.fit_transform(
  ..., ...
)  # les textes, puis les vecteurs déjà calculés
info = topic_model.get_topic_info()
print(len(info) - 1, "thèmes,", f"{np.mean(np.array(topics) == -1):.0%} hors thème")
info.head(16)

for topic in info.Topic[1:4]:
  print(topic, topic_model.get_topic(topic)[:8])
  for doc in topic_model.get_representative_docs(...)[:2]:
    print("   ", doc[:200])
```

### Solution

In [ ]:
topic_model = BERTopic(
  language="french",
  umap_model=UMAP(
    n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
  ),
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
  min_topic_size=50,
)
topics, _ = topic_model.fit_transform(docs, vectors)
info = topic_model.get_topic_info()
print(len(info) - 1, "thèmes,", f"{np.mean(np.array(topics) == -1):.0%} hors thème")
info.head(16)

In [ ]:
for topic in info.Topic[1:4]:
  print(topic, topic_model.get_topic(topic)[:8])
  for doc in topic_model.get_representative_docs(topic)[:2]:
    print("   ", doc[:200])

## 2. Ajuster les thèmes

- Relancez avec `min_topic_size=20`, puis `200` : combien de thèmes, quelle part hors thème ?
- Avec le modèle de départ, réaffectez les passages hors thème à leur thème le plus proche (`reduce_outliers`), puis mettez à jour les descriptions (`update_topics`).
- Repérez deux thèmes qui parlent de la même chose et fusionnez-les (`merge_topics`).

**Documentation utile**

- [Réduire les hors-thème](https://maartengr.github.io/BERTopic/getting_started/outlier_reduction/outlier_reduction.html)
- [`BERTopic.update_topics`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.update_topics)
- [`BERTopic.merge_topics`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.merge_topics)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- relancer avec une autre taille : dans une boucle sur `(20, 200)`, un nouveau `BERTopic` de même configuration, avec `min_topic_size=size` (sans écraser `topic_model`, le modèle de départ)
- `topic_model.reduce_outliers(docs, topics, strategy="embeddings", embeddings=vectors)` renvoie les nouveaux thèmes des passages
- `topic_model.update_topics(docs, topics=new_topics, vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10))` : recalcule les mots-clés après la réaffectation
- `topic_model.merge_topics(docs, [a, b])` : `a` et `b` sont deux thèmes à fusionner en un seul, choisis en lisant les mots-clés de `get_topic_info()`. Les numéros de thèmes changent après une fusion : `merge_topics` renumérote tous les thèmes et ne renvoie rien (`None`), reprenez donc `topics = topic_model.topics_` juste après

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for size in (20, 200):
  model = BERTopic(
    language="french",
    umap_model=UMAP(
      n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
    ),
    vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
    min_topic_size=...,
  )
  found, _ = model.fit_transform(docs, vectors)
  print(
    f"min_topic_size={size} : {len(set(found)) - 1} thèmes, {np.mean(np.array(found) == -1):.0%} hors thème"
  )

new_topics = topic_model.reduce_outliers(docs, topics, strategy=..., embeddings=vectors)
topic_model.update_topics(docs, topics=new_topics, vectorizer_model=...)
print(f"{np.mean(np.array(new_topics) == -1):.0%} hors thème après réaffectation")
topics = new_topics

topic_model.merge_topics(docs, [..., ...])  # deux thèmes qui parlent de la même chose
topics = topic_model.topics_  # les numéros ont changé
```

### Solution

In [ ]:
for size in (20, 200):
  model = BERTopic(
    language="french",
    umap_model=UMAP(
      n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=0
    ),
    vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
    min_topic_size=size,
  )
  found, _ = model.fit_transform(docs, vectors)
  print(
    f"min_topic_size={size} : {len(set(found)) - 1} thèmes, {np.mean(np.array(found) == -1):.0%} hors thème"
  )

In [ ]:
new_topics = topic_model.reduce_outliers(
  docs, topics, strategy="embeddings", embeddings=vectors
)
topic_model.update_topics(
  docs,
  topics=new_topics,
  vectorizer_model=CountVectorizer(stop_words=STOP_WORDS, min_df=10),
)
print(f"{np.mean(np.array(new_topics) == -1):.0%} hors thème après réaffectation")
topics = new_topics

# Ici, deux thèmes parlent des législatives d'avril-mai 1914 : les candidats d'un côté,
# les tableaux de résultats, abrégés, de l'autre. Leurs numéros changent avec les
# versions des bibliothèques : on les retrouve par leur mot-clé commun, « voix ».
elections = [
  topic
  for topic in topic_model.get_topics()
  if "voix" in [word for word, _ in topic_model.get_topic(topic)]
]
for topic in elections:
  print(topic, [word for word, _ in topic_model.get_topic(topic)[:8]])

topic_model.merge_topics(docs, elections)
topics = topic_model.topics_  # la fusion a renuméroté les thèmes

## 3. Suivre les thèmes dans le temps

`topic_model.topics_over_time` compte les passages de chaque thème par semaine ; `topic_model.visualize_topics_over_time` les trace. Affichez les thèmes qui parlent de la guerre, de Jaurès et du procès Caillaux : écrivez une fonction `topic_with(word)` qui renvoie le thème dont la plus grande part des passages contient `word`. Que se passe-t-il entre juillet et août 1914 ?

**Documentation utile**

- [Les thèmes dans le temps](https://maartengr.github.io/BERTopic/getting_started/topicsovertime/topicsovertime.html)
- [`BERTopic.topics_over_time`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.topics_over_time)
- [`BERTopic.visualize_topics_over_time`](https://maartengr.github.io/BERTopic/api/bertopic.html#bertopic._bertopic.BERTopic.visualize_topics_over_time)
- [`Series.idxmax`](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `topic_model.topics_over_time(docs, dates, nr_bins=52)` : 52 intervalles, une semaine chacun sur un an
- `topic_model.visualize_topics_over_time(over_time, topics=chosen)` : `topics=` est la liste des thèmes à tracer
- `topic_with(word)` : parmi les passages qui contiennent le mot (`word in doc.lower()`), le nombre de passages de chaque thème (`pd.Series(topics)[masque].value_counts()`), divisé par la taille totale de ce thème (`pd.Series(topics).value_counts()`), sans le thème −1 (`.drop(-1, errors="ignore")`)
- `Series.idxmax()` : l'étiquette de la plus grande valeur, ici le numéro du thème ; essayez `"guerre"`, `"jaurès"` et `"caillaux"`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
over_time = topic_model.topics_over_time(docs, dates, nr_bins=...)


def topic_with(word: str) -> int:
  mentions = pd.Series(topics)[[word in doc.lower() for doc in docs]].value_counts()
  share = (mentions / pd.Series(topics).value_counts()).drop(-1, errors="ignore")
  return int(share.idxmax())


chosen = [topic_with(word) for word in ("guerre", "jaurès", ...)]
print(chosen)
topic_model.visualize_topics_over_time(over_time, topics=...)
```

### Solution

In [ ]:
over_time = topic_model.topics_over_time(docs, dates, nr_bins=52)


def topic_with(word: str) -> int:
  mentions = pd.Series(topics)[[word in doc.lower() for doc in docs]].value_counts()
  share = (mentions / pd.Series(topics).value_counts()).drop(-1, errors="ignore")
  return int(share.idxmax())


chosen = [topic_with(word) for word in ("guerre", "jaurès", "caillaux")]
print(chosen)
topic_model.visualize_topics_over_time(over_time, topics=chosen)

# En mars, Mme Caillaux tue Gaston Calmette, le directeur du Figaro ; son procès occupe
# juillet. Jaurès est assassiné le 31 juillet. À partir d'août, la guerre et les récits du
# front dominent.